<a href="https://colab.research.google.com/github/adithyaashok10/AI-ML-Intership/blob/main/Sentimental_Analysis_Day07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("/content/IMDB Dataset.csv", encoding='latin1', on_bad_lines='skip', engine='python')

print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
df['sentiment'] = df['sentiment'].str.lower().str.strip().map({
    'positive': 1,
    'negative': 0
})

In [ ]:
negation_words = [
    "not good", "not bad", "not great", "don't like "
    "didn't like", "never liked", "wasn't good",
    "isn't good", "no good"
]

In [ ]:
import re

def clean_text(text):
  text = text.lower()

  text = re.sub(r"[^a-zA-Z\s']"," ", text)

  for phrase in negation_words:
    text = text.replace(phrase, phrase.replace(" ", "_")) # Changed to underscore for clarity

  return text

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(x_train)

x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

x_train_padded = pad_sequences(x_train_seq, maxlen=max_len, padding='post', truncating='post')
x_test_padded = pad_sequences(x_test_seq, maxlen=max_len, padding='post', truncating='post')



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(vocab_size ,128 , input_length = max_len),
    LSTM(128, dropout = 0.3, recurrent_dropout = 0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(x_train_padded, y_train, epochs = 5, batch_size = 64, validation_split = 0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 400s 790ms/step - accuracy: 0.5126 - loss: 0.6927 - val_accuracy: 0.5239 - val_loss: 0.6926
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 393s 786ms/step - accuracy: 0.5623 - loss: 0.6625 - val_accuracy: 0.5512 - val_loss: 0.6784
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 394s 788ms/step - accuracy: 0.6266 - loss: 0.6123 - val_accuracy: 0.5405 - val_loss: 0.6842
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 393s 786ms/step - accuracy: 0.6735 - loss: 0.5876 - val_accuracy: 0.7735 - val_loss: 0.5653
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 439s 780ms/step - accuracy: 0.8302 - loss: 0.4079 - val_accuracy: 0.8529 - val_loss: 0.3665


In [ ]:
loss, acc = model.evaluate(x_test_padded, y_test)
print("Test Accuracy:", acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 32s 101ms/step - accuracy: 0.8481 - loss: 0.3688
Test Accuracy: 0.8481000065803528


In [19]:
def predict_sentiment(review):
  review = clean_text(review)
  review_seq = tokenizer.texts_to_sequences([review])
  review_padded = pad_sequences(review_seq, maxlen=max_len, padding='post', truncating='post')
  prediction = model.predict(review_padded)[0][0]
  print("\nReview : ", review)
  print("Score : ", prediction)
  if prediction > 0.5:
    print("Sentiment : Positive👌")
  else:
    print("Sentiment : Negative😒")


In [20]:
predict_sentiment("This movie is super")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step

Review :  this movie is super
Score :  0.6479276
Sentiment : Positive👌
